# 体積ハカルくん - Google Colab試作版

iPhoneで対象物の周囲を撮影した動画から、複数方向の輪郭を重ねた**視体積（visual hull）**を作り、体積を `cm³` / `mL` で推定します。

今回の改良版では、背景まで緑色になった明らかな不良マスクを自動で除外し、使用フレームを手動でも指定できます。結果には推定した縦・横・高さと、高さごとの断面積も保存します。

## この試作版の対象

- 静止している、不透明な小物
- 深い凹みが少ない形状
- A4マーカーボード内に収まる対象
- 背景と色・輪郭を区別しやすい対象

> **重要:** 自動判定は明らかな失敗を減らす補助機能です。見えない凹みは埋まった形として計算され、輪郭の誤りが残ると体積は大きくずれます。精密測定、診断、安全性・品質などの重要な判断には使用しないでください。


## 撮影前の準備

1. [A4マーカーボードPDF](https://satorumuro.github.io/hakarukun-web/volume/volume-marker-board-a4.pdf)を**100%・実際のサイズ**で印刷します。
2. 下部の確認線が正確に100 mmであることを定規で確認します。
3. 対象物をボード中央に置き、紙を平らに固定します。
4. ズームを変えず、ボードをなるべく画面に残したまま、10〜20秒かけて一周撮影します。
5. 高さが分かるよう少し斜め上から撮り、対象物の全周を映します。
6. 対象物と似た色の背景、強い影、反射、手ぶれを避けます。

`最大対象物高さ` は実際の高さより20〜40 mmほど大きく設定してください。浅い物体を150 mmなど大きすぎる範囲で計算すると、誤った輪郭が高さ方向に残りやすくなります。


In [ ]:
# 1. 必要なライブラリを準備します（初回は数分かかります）
%pip install -q "numpy<2.3" "rembg==2.0.67" "onnxruntime==1.22.1" "trimesh==4.8.1" "scikit-image==0.25.2" "plotly==6.3.0"
%pip uninstall -q -y opencv-python-headless
%pip install -q --force-reinstall --no-deps "opencv-contrib-python-headless==4.12.0.88"


In [ ]:
# 2. 体積計算パイプラインを読み込みます
from pathlib import Path
import math
import shutil
import urllib.request

module_path = Path('/content/volume_pipeline.py')
module_urls = [
    'https://raw.githubusercontent.com/SatoruMuro/hakarukun-web/main/colab/volume_pipeline.py',
    'https://raw.githubusercontent.com/SatoruMuro/hakarukun-web/agent/volume-colab-prototype/colab/volume_pipeline.py',
]
for url in module_urls:
    try:
        urllib.request.urlretrieve(url, module_path)
        if module_path.stat().st_size > 5000:
            print('Loaded:', url)
            break
    except Exception:
        continue
else:
    raise RuntimeError('体積計算モジュールを取得できませんでした。しばらくしてから再実行してください。')

from volume_pipeline import (
    BoardSpec, VolumeBounds, calibrate_camera_from_board,
    camera_orbit_coverage_degrees, carve_visual_hull, cross_section_rows,
    estimate_board_poses, extract_video_frames, save_contact_sheet,
    save_volume_outputs, segment_pose_frames_with_rembg, select_pose_frames,
)


In [ ]:
# 3. iPhoneで撮影した動画を選びます
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('動画が選択されていません。')
video_name = next(iter(uploaded))
video_path = Path('/content') / video_name
print('動画:', video_path.name, f'({video_path.stat().st_size / 1024 / 1024:.1f} MB)')


In [ ]:
# 4. 測定条件を設定します
TARGET_FRAMES = 40 # @param {type:"slider", min:20, max:60, step:4}
MAX_OBJECT_HEIGHT_MM = 60 # @param {type:"slider", min:20, max:220, step:5}
BOARD_MARGIN_MM = 25 # @param {type:"slider", min:10, max:50, step:5}
VOXEL_SIZE_MM = 2 # @param {type:"slider", min:2, max:8, step:1}
SUPPORT_RATIO = 0.88 # @param {type:"slider", min:0.70, max:0.98, step:0.01}
KNOWN_VOLUME_CM3 = 0 # @param {type:"number"}

spec = BoardSpec()
bounds = VolumeBounds(
    x_min_m=BOARD_MARGIN_MM / 1000,
    x_max_m=spec.width_m - BOARD_MARGIN_MM / 1000,
    y_min_m=BOARD_MARGIN_MM / 1000,
    y_max_m=spec.height_m - BOARD_MARGIN_MM / 1000,
    height_m=MAX_OBJECT_HEIGHT_MM / 1000,
)
bounds.validate(spec)
print(f'探索範囲: {(bounds.x_max_m-bounds.x_min_m)*1000:.0f} × {(bounds.y_max_m-bounds.y_min_m)*1000:.0f} × {bounds.height_m*1000:.0f} mm')
print(f'ボクセル: {VOXEL_SIZE_MM} mm / 輪郭一致率: {SUPPORT_RATIO:.0%}')
if KNOWN_VOLUME_CM3 > 0:
    print(f'検証用の既知体積: {KNOWN_VOLUME_CM3:.2f} cm³')


In [ ]:
# 5. 動画から手ぶれの少ないフレームを抽出します
from IPython.display import Image, display
work_dir = Path('/content/volume_hakarukun_work')
work_dir.mkdir(exist_ok=True)
frames = extract_video_frames(video_path, work_dir / 'frames', target_frames=TARGET_FRAMES)
sheet = save_contact_sheet(frames, work_dir / 'frames_contact_sheet.jpg')
print(f'{len(frames)}フレームを抽出しました。ボードと対象物が一周を通して見えるか確認してください。')
display(Image(filename=str(sheet), width=960))


In [ ]:
# 6. マーカーボードからカメラと各フレームの位置を求めます
camera_matrix, distortion, reprojection_error, calibrated_paths = calibrate_camera_from_board(frames, spec)
poses = estimate_board_poses(frames, camera_matrix, distortion, spec)
print(f'姿勢を推定できたフレーム: {len(poses)} / {len(frames)}')
print(f'再投影誤差: {reprojection_error:.3f} px')
if reprojection_error > 2.0:
    print('⚠️ カメラ推定誤差が大きめです。手ぶれやボードの折れ・反りを確認してください。')


In [ ]:
# 7. 対象領域を自動抽出し、明らかな失敗を判定します
segmented = segment_pose_frames_with_rembg(
    poses, work_dir / 'masks', camera_matrix, distortion, bounds
)
qualities = [item.mask_quality for item in segmented]
mask_sheet = save_contact_sheet(
    [item.image_path for item in segmented],
    work_dir / 'mask_contact_sheet.jpg',
    [item.mask_path for item in segmented],
    qualities=qualities,
)
auto_rejected = [item for item in segmented if item.mask_quality and not item.mask_quality.accepted]
print(f'自動判定: OK {len(segmented)-len(auto_rejected)} / 除外候補 {len(auto_rejected)}')
reason_labels = {
    'mask-too-small': '対象マスクが小さすぎる',
    'mask-too-large': '対象マスクが大きすぎる',
    'touches-search-border': '探索範囲の境界まで緑色',
    'touches-image-edge': '画像端まで緑色',
    'far-from-object-center': '対象中心から離れている',
    'area-outlier': '他フレームと面積が大きく異なる',
}
for item in auto_rejected:
    reasons = [reason_labels.get(reason, reason) for reason in item.mask_quality.reasons]
    print(f'  #{item.frame_index:02d}: {", ".join(reasons)}')
print('緑色が対象物だけを覆うか確認してください。緑枠=自動採用、赤枠=自動除外です。')
display(Image(filename=str(mask_sheet), width=960))


In [ ]:
# 8. 使用フレームを確定します
# 緑がボードや背景まで漏れているフレーム番号を、例: "3, 8-10, 17" のように入力します。
AUTO_REJECT_BAD_MASKS = True # @param {type:"boolean"}
EXCLUDE_FRAMES = "" # @param {type:"string"}

selected_frames, rejected_frames = select_pose_frames(
    segmented,
    excluded_frames=EXCLUDE_FRAMES,
    auto_reject_bad_masks=AUTO_REJECT_BAD_MASKS,
)
orbit_coverage = camera_orbit_coverage_degrees(selected_frames, bounds)
print(f'使用: {len(selected_frames)} / 除外: {len(rejected_frames)}')
print('使用番号:', [item.frame_index for item in selected_frames])
print(f'推定した周回カバー範囲: {orbit_coverage:.1f}°' if orbit_coverage is not None else '周回範囲を計算できませんでした。')
if len(selected_frames) < 8:
    raise RuntimeError('使用可能な輪郭が8枚未満です。撮影条件を改善して撮り直してください。')
if orbit_coverage is not None and orbit_coverage < 200:
    raise RuntimeError('対象物の反対側まで撮れていません。より完全に一周した動画でやり直してください。')
if orbit_coverage is not None and orbit_coverage < 280:
    print('⚠️ 一周のカバー範囲が狭めです。結果は参考値として扱ってください。')


In [ ]:
# 9. 採用した輪郭から3D形状を削り出し、体積を計算します
voxel_size_m = VOXEL_SIZE_MM / 1000
minimum_views = max(6, min(12, math.ceil(len(selected_frames) * 0.4)))
occupancy, axes, hit_count, view_count = carve_visual_hull(
    selected_frames, camera_matrix, distortion, bounds,
    voxel_size_m=voxel_size_m,
    support_ratio=SUPPORT_RATIO,
    minimum_views=minimum_views,
)
result_dir = Path('/content/volume_hakarukun_result')
if result_dir.exists():
    shutil.rmtree(result_dir)
result = save_volume_outputs(
    occupancy, axes, voxel_size_m, result_dir,
    len(selected_frames), minimum_views, SUPPORT_RATIO,
    accepted_frames=selected_frames,
    rejected_frames=rejected_frames,
    orbit_coverage_degrees=orbit_coverage,
    hit_count=hit_count,
    view_count=view_count,
)
shutil.copy2(mask_sheet, result_dir / 'mask_contact_sheet.jpg')

dimensions = result.dimensions
print(f'推定体積: {result.voxel_volume_cm3:,.2f} cm³（約 {result.voxel_volume_cm3:,.2f} mL）')
print(f'推定外形: {dimensions.length_mm:.1f} × {dimensions.width_mm:.1f} × {dimensions.height_mm:.1f} mm')
print(f'最大断面積: {dimensions.max_cross_section_cm2:.2f} cm²')
print(f'使用フレーム: {result.usable_frames} / ボクセル寸法: {result.voxel_size_mm} mm')
if KNOWN_VOLUME_CM3 > 0:
    error_percent = (result.voxel_volume_cm3 / KNOWN_VOLUME_CM3 - 1) * 100
    print(f'既知体積との差: {error_percent:+.1f}%')
for warning in result.quality_warnings:
    print('⚠️', warning)
print('注意:', result.warning)


In [ ]:
# 10. 3Dモデルと高さ別の断面積を表示します
import plotly.graph_objects as go
import trimesh

mesh = trimesh.load(result_dir / 'volume_model.stl', force='mesh')
vertices = mesh.vertices * 1000
faces = mesh.faces
figure = go.Figure(data=[go.Mesh3d(
    x=vertices[:, 0], y=vertices[:, 1], z=-vertices[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    color='#08775f', opacity=0.82, flatshading=False,
)])
figure.update_layout(
    title=f'推定体積 {result.voxel_volume_cm3:,.2f} cm³ / 高さ {result.dimensions.height_mm:.1f} mm',
    scene=dict(aspectmode='data', xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='高さ (mm)'),
    margin=dict(l=0, r=0, b=0, t=45),
)
figure.show()

sections = [(height, area) for height, area in cross_section_rows(occupancy, axes, voxel_size_m) if area > 0]
section_figure = go.Figure(go.Scatter(
    x=[height for height, _ in sections],
    y=[area for _, area in sections],
    mode='lines+markers',
    line=dict(color='#08775f', width=3),
))
section_figure.update_layout(
    title='高さ別の推定断面積',
    xaxis_title='ボードからの高さ (mm)',
    yaxis_title='断面積 (cm²)',
    margin=dict(l=50, r=20, b=50, t=45),
)
section_figure.show()


In [ ]:
# 11. 結果一式をZIPでダウンロードします
archive = shutil.make_archive('/content/volume_hakarukun_result', 'zip', result_dir)
files.download(archive)


## 結果の見方

- `result.json`: 推定体積、推定外形寸法、使用・除外フレーム、品質警告
- `mask_contact_sheet.jpg`: 番号付きの輪郭確認画像（緑枠=自動採用、赤枠=自動除外）
- `mask_quality.json`: 各フレームの自動判定値
- `cross_sections.csv`: ボードからの高さごとの断面積
- `volume_model.glb` / `volume_model.stl`: 3Dモデル
- `occupancy_grid.npz`: 輪郭支持数を含む再解析用3Dデータ

外形の高さが実寸より大きい、探索範囲の上端に接する、または断面積が上方まで不自然に続く場合は、体積値を採用しないでください。マスク画像を見直して不良フレームを手動除外し、それでも改善しなければ背景・照明・撮影角度を変えて撮り直します。

同じ対象を3回以上撮影し、既知体積との差（正確さ）と測定間のばらつき（再現性）を別々に確認してください。
